In [9]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
client = Anthropic()
model = "claude-haiku-4-5-20251001"

In [8]:
#import methods
# to keep track of the conversation history from the user
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

# to keep track of the conversation history from the api
def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system_prompt = None, temperature = 1.0, stop_sequences = None):
    
    # base parameters
    parameters = {
        "model": model,
        "max_tokens": 600,
        "messages": messages,
        "temperature": temperature
    }
    
    if system_prompt:
        parameters["system_prompt"] = system_prompt
    
    if stop_sequences is not None:
        parameters["stop_sequences"] = stop_sequences

    response = client.messages.create(**parameters)

    assistant_message = response.content[0].text.strip()
    add_assistant_message(messages, assistant_message)
    return assistant_message

In [10]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json") # prefix message
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [12]:
data_set = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(data_set, f, indent = 2)

In [13]:
def run_prompt(test_case):
    # merge the prompt and the test case input then return the resulting response
    prompt = f"""
please solve the following tasks:
{test_case["task"]}
"""
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [ ]:
def run_test_case(test_case):
    #Call runt_prompt, then grade the result
    output = run_prompt(test_case)

    # TODO-Grading
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }